~ 45 GB of RAM is needed in your instance to successfully run all cells; this is due to the raster production functions near the bottom of the notebook.

In [ ]:
from datetime import datetime
from pprint import pprint
from satellogic import satellogic_v2
from shared_utils import s3utils

In [ ]:
###### CONFIG #######
bucket = "csda-data-vendor-satellogic"
prefix = "disasters"
upload_bucket = "s3://nasa-disasters/test_uploads"

In [ ]:
# Display all filles within the bucket
s3utils.retrieve_s3_file_list(bucket, prefix)

In [ ]:
# Collect the lists of metadata and image files within the bucket based on date (any date works,
# the function will find the nearest available date within the bucket) and data level
metadata, tifs = satellogic_v2.retrieve_satellogic_resources(datetime(2026, 1, 29), "L1D", bucket, prefix)

pprint(metadata)
pprint(tifs)

In [ ]:
# These functions directly accept the lists of image and/or metadata files and filter out the ones
# they actually need to function; for plotting purposes, they also return their output filepath
satellogic_v2.getSolarZenithAngle(metadata)

In [ ]:
# By default, all generated images save to the s3_temp folder that the base images
# are downloaded to, but this can be changed with the save_location argument
TC = satellogic_v2.genTrueColor(tifs, metadata, save_location = "./s3_temp")

In [ ]:
IR = satellogic_v2.gencolorIR(tifs, metadata)

In [ ]:
NDVI = satellogic_v2.genNDVI(tifs, metadata)

In [ ]:
NDWI = satellogic_v2.genNDWI(tifs, metadata)

In [ ]:
for filepath in [TC, IR, NDVI, NDWI]:
    cog_filepath = s3utils.convert_to_cog(filepath, filepath.replace(".tif", "_cog.tif"))
    s3_upload_filepath = s3utils.build_flat_s3_uri(upload_bucket, cog_filepath.split("/")[-1])
    print(s3_upload_filepath)
    s3utils.upload_file_to_s3(cog_filepath, s3_upload_filepath)

In [ ]:
# This cleans out the s3_temp directory currently used for storing the 
# downloaded base geotiffs and produced rasters
s3utils.remove_s3_temp()